# Demo 1: Happy Path (Non-Technical Walkthrough)\n\n## Objective\nShow a full successful flow:\n1. Check API health\n2. Register and login a user\n3. Create a product\n4. Place an order\n5. List orders for that user\n\n## Before you start\nRun these commands in terminal: \n- `make up`\n- `make health`

In [ ]:
BASE_URL = "http://localhost:8000"\nBASE_URL

In [ ]:
import json\nimport uuid\n\nimport requests\n\ndef show_response(label, response):\n    print(f"\n=== {label} ===")\n    print("Status:", response.status_code)\n    try:\n        print(json.dumps(response.json(), indent=2, ensure_ascii=False))\n    except Exception:\n        print(response.text)

## 1) Health Check\nExpected: **200** with `{"status":"ok"}`

In [ ]:
r = requests.get(f"{BASE_URL}/health", timeout=15)\nshow_response("Health", r)

## 2) Register and Login\nWe generate unique user data so this notebook can be re-run safely.

In [ ]:
run_id = uuid.uuid4().hex[:8]\nemail = f"demo_happy_{run_id}@example.com"\npassword = "StrongPass123"\n\nregister_payload = {"email": email, "password": password}\nr_register = requests.post(f"{BASE_URL}/auth/register", json=register_payload, timeout=15)\nshow_response("Register", r_register)\n\nr_login = requests.post(f"{BASE_URL}/auth/login", json=register_payload, timeout=15)\nshow_response("Login", r_login)\n\nlogin_json = r_login.json()\ntoken = login_json.get("access_token")\nheaders = {"Authorization": f"Bearer {token}"}\nprint("Token captured:", bool(token))

## 3) Create Product\nExpected: **201** and product with `status` = `in_stock` (quantity > 0).

In [ ]:
product_id = f"prod_{run_id}"\nproduct_payload = {\n    "product_id": product_id,\n    "product_name": "Demo Notebook Product",\n    "quantity": 5\n}\nr_product = requests.post(f"{BASE_URL}/products", json=product_payload, timeout=15)\nshow_response("Create Product", r_product)

## 4) Place Order (Authenticated)\nExpected: **201** with message `order_placed`.

In [ ]:
order_payload = {"product_id": product_id, "quantity": 2}\nr_order = requests.post(f"{BASE_URL}/orders", json=order_payload, headers=headers, timeout=15)\nshow_response("Place Order", r_order)

## 5) List Orders for Logged User\nExpected: **200** and at least one order from this notebook run.

In [ ]:
r_orders = requests.get(f"{BASE_URL}/orders", headers=headers, timeout=15)\nshow_response("List Orders", r_orders)

## Business Outcome (Plain English)\n- The user authenticated successfully.\n- The order was accepted and recorded.\n- Product stock was reduced atomically.\n- Orders are scoped to the logged-in user account.